# Limpieza de Leads Reales

In [3]:
import pandas as pd

# Ajusta esta ruta si el archivo no esta en la misma carpeta que este notebook
archivo_entrada = "Analisis GAC Full7agosto 2026(Leads Reales).csv"

# header=None porque la primera fila del archivo es un titulo ("Efectivadad de Leads"),
# no el encabezado real de la tabla
raw = pd.read_csv(archivo_entrada, encoding="utf-8-sig", header=None)

# el encabezado real de la tabla mensual esta en la fila 3 del archivo (indice 2),
# y los datos de Enero 2024 a Mayo 2026 van de la fila 4 a la 32 (indice 3 a 31)
encabezado = raw.iloc[2, 0:12].str.strip().tolist()
df = raw.iloc[3:32, 0:12].copy()
df.columns = encabezado
df = df.reset_index(drop=True)

print("Forma de la tabla mensual:", df.shape)
df.head()

Forma de la tabla mensual: (29, 12)


,Año,Mes,Total,Ilocalizable,No Contesta,D. Incorrectos,Duplicados,Efectivos,Abiertos,Ventas,Conversion,Efectividad de Leads
0,2024,Enero,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,Febrero,543,108,33,0,1,389,1,12,3.1%,72%
2,NaN,Marzo,178,37,13,0,0,128,1,7,5.5%,72%
3,NaN,Abril,416,79,46,3,0,288,1,8,2.8%,69%
4,NaN,Mayo,422,33,100,17,0,274,5,13,4.7%,65%


In [4]:
df["Año"] = df["Año"].ffill().astype(int)
df["Mes"] = df["Mes"].str.strip()

columnas_conteo = ["Total", "Ilocalizable", "No Contesta", "D. Incorrectos", "Duplicados", "Efectivos", "Abiertos", "Ventas"]
for col in columnas_conteo:
    df[col] = df[col].str.replace(",", "", regex=False)
    df[col] = pd.to_numeric(df[col], errors="coerce")

columnas_porcentaje = ["Conversion", "Efectividad de Leads"]
for col in columnas_porcentaje:
    df[col] = pd.to_numeric(df[col].str.replace("%", "", regex=False), errors="coerce")

# se renombran para dejar claro que ya son valores porcentuales (ej. 3.1 = 3.1%)
df = df.rename(columns={"Conversion": "Conversion (%)", "Efectividad de Leads": "Efectividad de Leads (%)"})

print(df.dtypes)

Año                           int64
Mes                             str
Total                       float64
Ilocalizable                float64
No Contesta                 float64
D. Incorrectos              float64
Duplicados                  float64
Efectivos                   float64
Abiertos                    float64
Ventas                      float64
Conversion (%)              float64
Efectividad de Leads (%)    float64
dtype: object


In [5]:
print("Forma final:", df.shape)
print("Filas duplicadas:", df.duplicated().sum())

archivo_salida = "Leads_Reales_Limpia.csv"
df.to_csv(archivo_salida, index=False, encoding="utf-8-sig")
print("Archivo guardado como:", archivo_salida)

df.head()

Forma final: (29, 12)
Filas duplicadas: 0
Archivo guardado como: Leads_Reales_Limpia.csv


,Año,Mes,Total,Ilocalizable,No Contesta,D. Incorrectos,Duplicados,Efectivos,Abiertos,Ventas,Conversion (%),Efectividad de Leads (%)
0,2024,Enero,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024,Febrero,543.0,108.0,33.0,0.0,1.0,389.0,1.0,12.0,3.1,72.0
2,2024,Marzo,178.0,37.0,13.0,0.0,0.0,128.0,1.0,7.0,5.5,72.0
3,2024,Abril,416.0,79.0,46.0,3.0,0.0,288.0,1.0,8.0,2.8,69.0
4,2024,Mayo,422.0,33.0,100.0,17.0,0.0,274.0,5.0,13.0,4.7,65.0
